# Step 1 — Intent classification (IndoBERT-base-p2)

Labels already exist in `data/raw/intent_dataset.jsonl` (10-way, 200/class, balanced) — no labeling stage needed, unlike NER/disfluency. Fine-tunes `indobenchmark/indobert-base-p1` (`AutoModelForSequenceClassification`) on `text_normalized` (punctuation-stripped, fillers still present — matches what `4.1`/`4.2`'s NER pipeline already consumes) from `data/normalized/intent_dataset_normalized.jsonl`. `p1` chosen over `p2` for smaller/faster inference, since intent classification runs on every utterance in the live pipeline.

In [1]:
import json
from collections import Counter
from pathlib import Path

import numpy as np
import torch

SEED = 42
MAX_LENGTH = 64
MODEL_NAME = "indobenchmark/indobert-base-p1"
SRC = Path("../data/normalized/intent_dataset_normalized.jsonl")
OUT_DIR = Path("../data/intent")

## Step 2 — Load and inspect labels

Synthetic disfluency augmentation (`3.1`'s 60 added `FS`/`RP`/`RM` examples) landed here too, all tagged `order_create` — mild skew (260 vs 200 for other classes) but not severe enough to need weighted loss.

In [2]:
rows = [json.loads(line) for line in SRC.open(encoding="utf-8")]
print(f"{len(rows)} records loaded")

intent_counts = Counter(r["intent"] for r in rows)
for intent, count in intent_counts.most_common():
    print(f"  {intent:22s}: {count}")

2060 records loaded
  order_create          : 260
  order_add_item        : 200
  order_remove_item     : 200
  order_cancel          : 200
  order_modify_quantity : 200
  menu_inquiry          : 200
  confirm               : 200
  deny                  : 200
  repeat_request        : 200
  chitchat_oos          : 200


In [3]:
label_list = sorted(intent_counts.keys())
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for label, i in label2id.items()}
print(label2id)

{'chitchat_oos': 0, 'confirm': 1, 'deny': 2, 'menu_inquiry': 3, 'order_add_item': 4, 'order_cancel': 5, 'order_create': 6, 'order_modify_quantity': 7, 'order_remove_item': 8, 'repeat_request': 9}


## Step 3 — Tokenize

In [4]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def encode(text: str) -> dict:
    encoding = tokenizer(
        text,
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
    )
    return {"input_ids": encoding["input_ids"], "attention_mask": encoding["attention_mask"]}


encoded_rows = []
for r in rows:
    enc = encode(r["text_normalized"])
    encoded_rows.append({
        "id": r["id"],
        "input_ids": enc["input_ids"],
        "attention_mask": enc["attention_mask"],
        "labels": label2id[r["intent"]],
    })

print(f"{len(encoded_rows)} records encoded")

2060 records encoded


## Step 4 — Stratified 80/10/10 split

Stratified by `intent` to keep the 10-way balance across splits, same convention as `scripts/split_disfluency_repair_dataset.py`.

In [5]:
from sklearn.model_selection import train_test_split

VAL_FRAC = 0.1
TEST_FRAC = 0.1

labels_strat = [r["intent"] for r in rows]
train, rest = train_test_split(
    encoded_rows, test_size=VAL_FRAC + TEST_FRAC, stratify=labels_strat, random_state=SEED
)
rest_strat = [label_list[r["labels"]] for r in rest]
val, test = train_test_split(
    rest, test_size=TEST_FRAC / (VAL_FRAC + TEST_FRAC), stratify=rest_strat, random_state=SEED
)

print(f"train={len(train)} val={len(val)} test={len(test)}")

train=1648 val=206 test=206


In [6]:
def write_jsonl(path: Path, records: list[dict]) -> None:
    with path.open("w") as f:
        for r in records:
            f.write(json.dumps(r) + "\n")


OUT_DIR.mkdir(parents=True, exist_ok=True)
write_jsonl(OUT_DIR / "train.jsonl", train)
write_jsonl(OUT_DIR / "val.jsonl", val)
write_jsonl(OUT_DIR / "test.jsonl", test)
json.dump(label2id, (OUT_DIR / "label_map.json").open("w"), indent=2)

print("wrote", OUT_DIR)

wrote ../data/intent


## Step 5 — Load splits as `datasets.Dataset`

In [7]:
from datasets import Dataset


def load_split(jsonl_path: Path) -> Dataset:
    records = [json.loads(line) for line in jsonl_path.open(encoding="utf-8")]
    return Dataset.from_list([
        {"input_ids": r["input_ids"], "attention_mask": r["attention_mask"], "labels": r["labels"]}
        for r in records
    ])


train_dataset = load_split(OUT_DIR / "train.jsonl")
val_dataset = load_split(OUT_DIR / "val.jsonl")
test_dataset = load_split(OUT_DIR / "test.jsonl")

print(train_dataset)
print(val_dataset)
print(test_dataset)

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 1648
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 206
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 206
})


## Step 6 — Model architecture

Full fine-tune of `indobenchmark/indobert-base-p1` via `AutoModelForSequenceClassification`.

In [8]:
from transformers import AutoModelForSequenceClassification

device = (
    torch.device("mps") if torch.backends.mps.is_available()
    else torch.device("cuda") if torch.cuda.is_available()
    else torch.device("cpu")
)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
)
model.to(device)

n_params = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"device          : {device}")
print(f"total params    : {n_params:,}")
print(f"trainable params: {n_trainable:,}  (full fine-tune — should equal total)")

[transformers] You passed `num_labels=10` which is incompatible to the `id2label` map of length `5`.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: indobenchmark/indobert-base-p1
Key               | Status  | 
------------------+---------+-
classifier.bias   | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


device          : mps
total params    : 124,449,034
trainable params: 124,449,034  (full fine-tune — should equal total)


### Sanity check: forward pass on a real batch

In [9]:
sample_batch = train_dataset[:4]
batch_input_ids = torch.tensor(sample_batch["input_ids"], device=device)
batch_attention_mask = torch.tensor(sample_batch["attention_mask"], device=device)
batch_labels = torch.tensor(sample_batch["labels"], device=device)

model.eval()
with torch.no_grad():
    outputs = model(input_ids=batch_input_ids, attention_mask=batch_attention_mask, labels=batch_labels)

print(f"logits shape: {tuple(outputs.logits.shape)}  (expect (4, {len(label_list)}))")
assert outputs.logits.shape == (4, len(label_list))
print(f"sample loss: {outputs.loss.item():.4f}")

model.train()
print("Step 6 sanity checks passed.")

logits shape: (4, 10)  (expect (4, 10))
sample loss: 2.4752
Step 6 sanity checks passed.


## Step 7 — Training

Plain `Trainer` (unweighted) — class imbalance here is mild (1.3x at most), unlike the disfluency/NER stages, so the inverse-frequency weighting used there isn't warranted.

In [10]:
import evaluate

accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=preds, references=labels)
    f1 = f1_metric.compute(predictions=preds, references=labels, average="macro")
    return {"accuracy": acc["accuracy"], "f1_macro": f1["f1"]}

In [11]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="../models/indobert-intent",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    lr_scheduler_type="linear",
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    logging_strategy="epoch",
    report_to="none",
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [12]:
train_result = trainer.train()
train_result.metrics

/Users/michaeleko/Library/Caches/pypoetry/virtualenvs/ordo-ai-INBru7F4-py3.11/lib/python3.11/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,1.087414,0.049193,0.995146,0.995549
2,0.017782,0.010781,1.000000,1.000000
3,0.006978,0.007072,1.000000,1.000000
4,0.005133,0.006114,1.000000,1.000000
5,0.004457,0.005921,1.000000,1.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/michaeleko/Library/Caches/pypoetry/virtualenvs/ordo-ai-INBru7F4-py3.11/lib/python3.11/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

/Users/michaeleko/Library/Caches/pypoetry/virtualenvs/ordo-ai-INBru7F4-py3.11/lib/python3.11/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/michaeleko/Library/Caches/pypoetry/virtualenvs/ordo-ai-INBru7F4-py3.11/lib/python3.11/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'train_runtime': 280.8276,
 'train_samples_per_second': 29.342,
 'train_steps_per_second': 1.834,
 'total_flos': 271023852933120.0,
 'train_loss': 0.22435270455277082,
 'epoch': 5.0}

### Save the fine-tuned model

In [13]:
final_model_path = "../models/indobert-intent-final"
trainer.save_model(final_model_path)
tokenizer.save_pretrained(final_model_path)
print(f"Saved fine-tuned model -> {final_model_path}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved fine-tuned model -> ../models/indobert-intent-final


## Step 8 — Evaluation

Held-out **test** split, per-intent precision/recall/F1 via `classification_report` — not just overall accuracy, since `chitchat_oos`/`repeat_request` (semantically closer to other intents) are the likeliest weak spots.

In [14]:
test_predictions = trainer.predict(test_dataset)
test_logits, test_labels = test_predictions.predictions, test_predictions.label_ids
test_preds = np.argmax(test_logits, axis=-1)

print("overall test metrics:")
for k, v in test_predictions.metrics.items():
    print(f"  {k:25s}: {v}")

/Users/michaeleko/Library/Caches/pypoetry/virtualenvs/ordo-ai-INBru7F4-py3.11/lib/python3.11/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


overall test metrics:
  test_loss                : 0.014056839980185032
  test_accuracy            : 0.9951456310679612
  test_f1_macro            : 0.9949968730456534
  test_runtime             : 1.5789
  test_samples_per_second  : 130.473
  test_steps_per_second    : 8.234


In [15]:
from sklearn.metrics import classification_report

report = classification_report(
    test_labels, test_preds, target_names=label_list, digits=3, zero_division=0
)
print(report)

                       precision    recall  f1-score   support

         chitchat_oos      1.000     1.000     1.000        20
              confirm      1.000     1.000     1.000        20
                 deny      1.000     1.000     1.000        20
         menu_inquiry      1.000     1.000     1.000        20
       order_add_item      1.000     1.000     1.000        20
         order_cancel      1.000     1.000     1.000        20
         order_create      1.000     1.000     1.000        26
order_modify_quantity      0.952     1.000     0.976        20
    order_remove_item      1.000     0.950     0.974        20
       repeat_request      1.000     1.000     1.000        20

             accuracy                          0.995       206
            macro avg      0.995     0.995     0.995       206
         weighted avg      0.995     0.995     0.995       206



### Misclassified examples

In [16]:
test_records = [json.loads(line) for line in (OUT_DIR / "test.jsonl").open(encoding="utf-8")]
rows_by_id = {r["id"]: r for r in rows}

n_mismatch = 0
shown = 0
for rec, true_id, pred_id in zip(test_records, test_labels, test_preds):
    if true_id == pred_id:
        continue
    n_mismatch += 1
    if shown < 15:
        text = rows_by_id[rec["id"]]["text_normalized"]
        print(f"  text={text!r}  true={id2label[true_id]}  pred={id2label[pred_id]}")
        shown += 1

print(f"\nMismatched records: {n_mismatch} / {len(test_records)}")

  text='es teh manis yang dua gelas tadi satu aja kurangin satu'  true=order_remove_item  pred=order_modify_quantity

Mismatched records: 1 / 206


## Step 9 — Inference

Loads the saved model fresh from disk to confirm standalone reload. `predict_intent(text)` tokenizes raw text and returns the predicted intent plus per-class probabilities.

In [17]:
from transformers import AutoModelForSequenceClassification as _AMFSC, AutoTokenizer as _AT
import torch.nn.functional as F

INFERENCE_MODEL_PATH = "../models/indobert-intent-final"

inference_tokenizer = _AT.from_pretrained(INFERENCE_MODEL_PATH)
inference_model = _AMFSC.from_pretrained(INFERENCE_MODEL_PATH)
inference_model.to(device)
inference_model.eval()

print(f"Loaded model from {INFERENCE_MODEL_PATH}")
print(f"id2label: {inference_model.config.id2label}")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loaded model from ../models/indobert-intent-final
id2label: {0: 'chitchat_oos', 1: 'confirm', 2: 'deny', 3: 'menu_inquiry', 4: 'order_add_item', 5: 'order_cancel', 6: 'order_create', 7: 'order_modify_quantity', 8: 'order_remove_item', 9: 'repeat_request'}


In [18]:
def predict_intent(text: str) -> dict:
    encoding = inference_tokenizer(
        text, truncation=True, max_length=MAX_LENGTH, return_tensors="pt"
    )
    encoding = {k: v.to(device) for k, v in encoding.items()}

    with torch.no_grad():
        logits = inference_model(**encoding).logits[0]
    probs = F.softmax(logits, dim=-1)
    pred_id = probs.argmax().item()

    return {
        "intent": id2label[pred_id],
        "confidence": probs[pred_id].item(),
        "probs": {id2label[i]: p.item() for i, p in enumerate(probs)},
    }

### Try it on fresh, unseen sentences

In [19]:
demo_sentences = [
    "eh gue mau pesen nasi goreng spesial dua porsi",
    "tambahin es teh manis satu lagi ya",
    "batalin ayam bakar yang tadi",
    "ganti jadi tiga porsi aja",
    "ada menu apa aja sih disini",
    "iya bener itu pesanannya",
    "gak usah jadi deh",
    "tolong ulang lagi pesanannya",
    "boleh minta tisu",
]

for text in demo_sentences:
    result = predict_intent(text)
    print(f"input     : {text}")
    print(f"intent    : {result['intent']}  (confidence={result['confidence']:.3f})")
    print()

input     : eh gue mau pesen nasi goreng spesial dua porsi
intent    : order_create  (confidence=0.996)

input     : tambahin es teh manis satu lagi ya
intent    : order_add_item  (confidence=0.994)

input     : batalin ayam bakar yang tadi
intent    : order_remove_item  (confidence=0.990)

input     : ganti jadi tiga porsi aja
intent    : order_modify_quantity  (confidence=0.992)

input     : ada menu apa aja sih disini
intent    : menu_inquiry  (confidence=0.993)

input     : iya bener itu pesanannya
intent    : confirm  (confidence=0.991)

input     : gak usah jadi deh
intent    : order_remove_item  (confidence=0.800)

input     : tolong ulang lagi pesanannya
intent    : repeat_request  (confidence=0.982)

input     : boleh minta tisu
intent    : chitchat_oos  (confidence=0.249)

